# Mutation Potentials Demo

Minimal demo of `DecoderLogProbPotential` and `compose_mutant_distribution` on a short peptide (`AAA`).

In [1]:
import torch
from pep_compass.models.encoder_decoder.hydramp_encoder_decoder import (
    HydrAMPEncoderDecoder,
)
from pep_compass.models.encoder_decoder.utils import decoder_jacobian
from pep_compass.local_enumeration.mutation.utils import get_mutations_from_s_u_standard
from pep_compass.local_enumeration.mutation.mutation_potentials import (
    DecoderLogProbPotential,
    compose_mutant_distribution,
)

In [2]:
device = torch.device("cpu")
hydramp = HydrAMPEncoderDecoder(
    jacobian_mode="approx",
    jacobian_eps=1e-6,
    field_eps=1e-6,
    device=device,
)

In [3]:
peptide = "AAA"

z = hydramp.encode_peptides([peptide])
print(f"Parent: {peptide}")
print(f"Latent shape: {z.shape}")

Parent: AAA
Latent shape: torch.Size([1, 64])


In [4]:
jac = decoder_jacobian(
    lambda x: hydramp.decoder_forward(x, softmax=False, flatten=True),
    z,
    jacobian_fn_mode="approx",
    jacobian_fn_kwargs={"jacobian_eps": 1e-6},
)
U, S, _ = torch.linalg.svd(jac, full_matrices=False)
print(f"U shape: {U.shape}, S shape: {S.shape}")

U shape: torch.Size([1, 525, 64]), S shape: torch.Size([1, 64])


In [5]:
mutations = get_mutations_from_s_u_standard(
    s=S[0].detach().cpu().numpy(),
    u=U[0].detach().cpu().numpy(),
    max_len=25,
    alphabet_size=21,
    direction_significance_threshold=1e-4,
    min_number_of_directions=5,
    token_threshold=0.05,
)

alphabet = list(" ACDEFGHIKLMNPQRSTVWY")
print("Proposed mutations:")
for pos in sorted(mutations.keys()):
    aas = [alphabet[i] for i in mutations[pos]]
    print(f"  pos {pos}: indices {mutations[pos]}  ->  {aas}")

Proposed mutations:
  pos 0: indices [1, 3, 6, 11, 14, 15, 16, 18, 19, 20, 2, 3, 4, 8, 9, 10, 12, 15, 18, 19, 20, 1, 2, 3, 4, 9, 10, 12, 15, 19, 20, 2, 4, 5, 6, 9, 10, 12, 15, 16, 19]  ->  ['A', 'D', 'G', 'M', 'Q', 'R', 'S', 'V', 'W', 'Y', 'C', 'D', 'E', 'I', 'K', 'L', 'N', 'R', 'V', 'W', 'Y', 'A', 'C', 'D', 'E', 'K', 'L', 'N', 'R', 'W', 'Y', 'C', 'E', 'F', 'G', 'K', 'L', 'N', 'R', 'S', 'W']
  pos 1: indices [1, 2, 3, 4, 6, 10, 11, 14, 15, 20, 1, 2, 3, 4, 5, 6, 9, 11, 12, 15, 16, 18, 19, 20, 1, 2, 13, 16, 18, 19, 20, 1, 3, 4, 5, 6, 7, 10, 12, 15]  ->  ['A', 'C', 'D', 'E', 'G', 'L', 'M', 'Q', 'R', 'Y', 'A', 'C', 'D', 'E', 'F', 'G', 'K', 'M', 'N', 'R', 'S', 'V', 'W', 'Y', 'A', 'C', 'P', 'S', 'V', 'W', 'Y', 'A', 'D', 'E', 'F', 'G', 'H', 'L', 'N', 'R']
  pos 2: indices [2, 3, 4, 6, 7, 8, 9, 10, 12, 14, 15, 18, 19, 20, 1, 2, 3, 4, 7, 8, 9, 10, 11, 12, 14, 15, 17, 19, 20, 1, 2, 6, 9, 11, 14, 15, 16, 19, 1, 5, 6, 10, 11, 13, 15, 16]  ->  ['C', 'D', 'E', 'G', 'H', 'I', 'K', 'L', 'N', 'Q', 'R',

In [7]:
potential = DecoderLogProbPotential(encoder_decoder=hydramp)

df = compose_mutant_distribution(
    parent_peptide=peptide,
    mutations=mutations,
    potential=potential,
    include_parent_residue=True,
    top_k=20,
)

df

ValueError: broadcast dimensions too large.

In [ ]:
potentials_dict = potential.compute(peptide, mutations)
print("Raw per-position potentials (first 3 positions):")
for pos in sorted(potentials_dict.keys())[:3]:
    entries = {
        alphabet[aa]: f"{lp:.4f}"
        for aa, lp in sorted(potentials_dict[pos].items(), key=lambda x: -x[1])
    }
    print(f"  pos {pos}: {entries}")